# 04 — Fundamental Analysis Engine

Phase 4 of the Indian Equity AI project plan.

For each of the 5 stocks:
1. Download income statement, balance sheet, cash flow (Yahoo Finance)
2. Compute ratios: ROE, ROCE, Debt/Equity, Current Ratio, Interest Coverage
3. Compute growth metrics (revenue/EBITDA/EPS/FCF YoY and CAGR)
4. Pull snapshot valuation ratios (P/E, P/B, P/S, EV/EBITDA, dividend yield) and ownership %
5. Combine into a single Fundamental Score (0-100)

**Data-source note:** bank income statements (HDFCBANK) don't report EBIT/EBITDA or a
current-assets/current-liabilities split, so ROCE/current ratio/interest coverage are
NaN for it -- this is expected, not a bug. The Fundamental Score itself only depends on
ROE, Debt/Equity, revenue growth, and P/E, all of which are available for every stock,
including HDFCBANK.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.fundamentals.statements import download_statements
from src.fundamentals.ratios import (
    add_growth_metrics,
    build_annual_fundamentals,
    cagr,
    snapshot_valuation_and_ownership,
)
from src.fundamentals.score import fundamental_score

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
from src.config import UNIVERSE  # Phase 14: centralized, now 50 stocks

## Step 1-3: Download statements, compute ratios and growth metrics, per stock

In [2]:
# Phase 14 (50-stock scale-out): a per-stock try/except is now needed -- with only
# 5 hand-picked large caps, every stock had clean, complete Yahoo Finance data; across
# 50 (including a newly-demerged entity, TMPV), some Ticker fetches may fail outright.
# Failures are collected and reported, not silently swallowed, and don't abort the run.
annual_by_symbol = {}
snapshot_by_symbol = {}
fetch_failures = {}

for symbol in UNIVERSE:
    try:
        statements = download_statements(symbol)
        annual = build_annual_fundamentals(statements)
        annual = add_growth_metrics(annual)
        annual_by_symbol[symbol] = annual
        snapshot_by_symbol[symbol] = snapshot_valuation_and_ownership(statements["info"])
        print(symbol, "fiscal years:", [d.date() for d in annual.index])
    except Exception as exc:
        fetch_failures[symbol] = str(exc)
        print(symbol, "FAILED:", exc)

print()
print(f"{len(annual_by_symbol)}/{len(UNIVERSE)} stocks fetched successfully.")
if fetch_failures:
    print("Failed:", fetch_failures)

RELIANCE fiscal years: [datetime.date(2022, 3, 31), datetime.date(2023, 3, 31), datetime.date(2024, 3, 31), datetime.date(2025, 3, 31), datetime.date(2026, 3, 31)]


TCS fiscal years: [datetime.date(2023, 3, 31), datetime.date(2024, 3, 31), datetime.date(2025, 3, 31), datetime.date(2026, 3, 31)]


D:\Practise\Stock Market Analysis\src\fundamentals\ratios.py:69: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  annual[f"{col}_growth_yoy"] = annual[col].pct_change()


HDFCBANK fiscal years: [datetime.date(2022, 3, 31), datetime.date(2023, 3, 31), datetime.date(2024, 3, 31), datetime.date(2025, 3, 31), datetime.date(2026, 3, 31)]


D:\Practise\Stock Market Analysis\src\fundamentals\ratios.py:69: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  annual[f"{col}_growth_yoy"] = annual[col].pct_change()


...

BAJAJ-AUTO fiscal years: [datetime.date(2023, 3, 31), datetime.date(2024, 3, 31), datetime.date(2025, 3, 31), datetime.date(2026, 3, 31)]


TATACONSUM fiscal years: [datetime.date(2022, 3, 31), datetime.date(2023, 3, 31), datetime.date(2024, 3, 31), datetime.date(2025, 3, 31), datetime.date(2026, 3, 31)]


ONGC fiscal years: [datetime.date(2023, 3, 31), datetime.date(2024, 3, 31), datetime.date(2025, 3, 31), datetime.date(2026, 3, 31)]


HINDALCO fiscal years: [datetime.date(2022, 3, 31), datetime.date(2023, 3, 31), datetime.date(2024, 3, 31), datetime.date(2025, 3, 31), datetime.date(2026, 3, 31)]


DMART fiscal years: [datetime.date(2023, 3, 31), datetime.date(2024, 3, 31), datetime.date(2025, 3, 31), datetime.date(2026, 3, 31)]

50/50 stocks fetched successfully.


In [3]:
annual_by_symbol["RELIANCE"][
    ["revenue", "net_income", "roe", "roce", "debt_to_equity", "current_ratio", "revenue_growth_yoy"]
]

,revenue,net_income,roe,roce,debt_to_equity,current_ratio,revenue_growth_yoy
2022-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-03-31,8.778350e+12,6.670200e+11,0.093176,0.109730,0.467111,1.074677,NaN
2024-03-31,9.010640e+12,6.962100e+11,0.087741,0.112791,0.436232,1.183037,0.026462
2025-03-31,9.646930e+12,6.964800e+11,0.082600,0.107613,0.438301,1.100351,0.070615
2026-03-31,1.057219e+13,8.077500e+11,0.089350,0.115153,0.440251,1.097912,0.095912


In [4]:
annual_by_symbol["HDFCBANK"][
    ["revenue", "net_income", "roe", "roce", "debt_to_equity", "current_ratio", "revenue_growth_yoy"]
]

,revenue,net_income,roe,roce,debt_to_equity,current_ratio,revenue_growth_yoy
2022-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-03-31,1.205375e+12,4.954469e+11,0.170128,NaN,1.082272,NaN,NaN
2024-03-31,1.604968e+12,6.226566e+11,0.089625,NaN,1.164988,NaN,0.331510
2025-03-31,1.838641e+12,6.735083e+11,0.087732,NaN,0.954306,NaN,0.145593
2026-03-31,1.925668e+12,7.047934e+11,0.086294,NaN,0.813434,NaN,0.047332


## Step 3b: CAGR over the available fiscal-year window

In [5]:
cagr_rows = []
for symbol, annual in annual_by_symbol.items():
    cagr_rows.append(
        {
            "symbol": symbol,
            "revenue_cagr": cagr(annual["revenue"]),
            "eps_cagr": cagr(annual["eps_diluted"]),
            "fcf_cagr": cagr(annual["free_cash_flow"]),
        }
    )
cagr_table = pd.DataFrame(cagr_rows).set_index("symbol")
cagr_table

D:\Practise\Stock Market Analysis\src\fundamentals\ratios.py:82: RuntimeWarning: invalid value encountered in scalar power
  return (clean.iloc[-1] / clean.iloc[0]) ** (1 / years) - 1


,revenue_cagr,eps_cagr,fcf_cagr
symbol,,,
RELIANCE,0.063941,0.065859,NaN
TCS,0.058018,0.056944,0.072516
HDFCBANK,0.169014,0.010489,0.349581
ICICIBANK,0.181134,0.160501,NaN
INFY,0.034419,0.040584,0.137847
HINDUNILVR,0.023053,0.141130,0.030824
ITC,0.036465,0.022145,0.003025
SBIN,0.121870,0.134983,NaN
BHARTIARTL,0.148823,0.449472,0.248686


## Step 4-5: Build the latest-fiscal-year snapshot table and Fundamental Score

In [6]:
rows = []
for symbol in annual_by_symbol:  # only successfully-fetched stocks
    latest = annual_by_symbol[symbol].iloc[-1].to_dict()
    latest.update(snapshot_by_symbol[symbol])
    latest["symbol"] = symbol
    # Website audit (fundamentals metadata gap): record which fiscal year the
    # balance-sheet-basis ratios (roe/roce/debt_to_equity/...) come from, and
    # which day the valuation-basis ratios (pe_ratio/pb_ratio/...) were fetched --
    # these are two different "as of" bases, previously conflated with no period
    # label at all.
    latest["fiscal_year"] = str(annual_by_symbol[symbol].index[-1])
    latest["valuation_as_of"] = pd.Timestamp.now().normalize().date().isoformat()
    rows.append(latest)

fundamentals_snapshot = pd.DataFrame(rows).set_index("symbol")
fundamentals_snapshot["fundamental_score"] = fundamental_score(fundamentals_snapshot)
fundamentals_snapshot[
    ["fiscal_year", "roe", "debt_to_equity", "revenue_growth_yoy", "valuation_as_of", "pe_ratio", "fundamental_score"]
]

,fiscal_year,roe,debt_to_equity,revenue_growth_yoy,valuation_as_of,pe_ratio,fundamental_score
symbol,,,,,,,
RELIANCE,2026-03-31 00:00:00,0.089350,0.440251,0.095912,2026-10-01,21.503622,48.971152
TCS,2026-03-31 00:00:00,0.458877,0.105213,0.045812,2026-10-01,14.903701,74.612212
HDFCBANK,2026-03-31 00:00:00,0.086294,0.813434,0.047332,2026-10-01,15.490710,45.849225
ICICIBANK,2026-03-31 00:00:00,0.149308,0.571853,0.091307,2026-10-01,17.076227,53.648694
INFY,2026-03-31 00:00:00,0.338545,0.098815,0.045702,2026-10-01,12.813868,72.152792
HINDUNILVR,2026-03-31 00:00:00,0.308589,0.030325,0.052664,2026-10-01,41.832146,54.435386
ITC,2026-03-31 00:00:00,0.285343,0.033091,0.047687,2026-10-01,16.577286,67.379783
SBIN,2026-03-31 00:00:00,0.139732,1.219350,0.092085,2026-10-01,10.264227,49.246452
BHARTIARTL,2026-03-31 00:00:00,0.179094,1.310990,0.219600,2026-10-01,36.718914,39.340049


## Step 6: Save fundamentals

In [7]:
# Phase 14: a hard assert here would abort the whole run over one stock's data gap.
# Report which stocks (if any) lack a computable score instead of failing the notebook --
# an honest, visible gap is reported, not silently dropped or force-fabricated.
missing_score = fundamentals_snapshot[fundamentals_snapshot["fundamental_score"].isna()]
if not missing_score.empty:
    print(f"WARNING: {len(missing_score)} stock(s) have no computable Fundamental Score:",
          missing_score.index.tolist())

fundamentals_snapshot.to_csv(DATA_PROCESSED / "fundamentals_snapshot.csv")

annual_long = pd.concat(
    [df.assign(symbol=symbol) for symbol, df in annual_by_symbol.items()]
).reset_index(names="fiscal_year_end")
annual_long.to_parquet(DATA_PROCESSED / "fundamentals_annual.parquet", index=False)

print("Saved:", DATA_PROCESSED / "fundamentals_snapshot.csv")
print("Saved:", DATA_PROCESSED / "fundamentals_annual.parquet")

Saved: D:\Practise\Stock Market Analysis\data\processed\fundamentals_snapshot.csv
Saved: D:\Practise\Stock Market Analysis\data\processed\fundamentals_annual.parquet
